# Olist Brazilian E-commerce — Exploratory Data Analysis

**BizSentinel Portfolio Project**  
This notebook performs a structured EDA on the Olist Brazilian E-commerce dataset.  
The goal is to understand data quality, customer behavior, RFM distributions, review patterns, churn characteristics, and preliminary anomaly candidates — all of which will inform downstream ML pipelines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = '../data/01_raw/'
pd.set_option('display.max_columns', None)

---
## Section 1: Dataset Overview

**Key Question:** What data quality issues do we need to handle in preprocessing?

We load all 7 core CSV tables and inspect shapes, dtypes, and null percentages.

In [ ]:
# Load all 7 core CSV tables
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')

tables = {
    'customers': customers,
    'orders': orders,
    'order_items': items,
    'order_payments': payments,
    'order_reviews': reviews,
    'products': products,
    'sellers': sellers,
}

In [ ]:
# Print shape and dtypes for each table
for name, df in tables.items():
    print(f'\n{'='*60}')
    print(f'{name.upper()} — shape: {df.shape}')
    print(f'{'='*60}')
    print(df.dtypes.to_string())

In [ ]:
# Print % of null values per column for each table
for name, df in tables.items():
    nulls = df.isnull().mean() * 100
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'\n{name.upper()} — null columns (%):')
        for col, pct in nulls.items():
            print(f'  {col:45s} {pct:.2f}%')
    else:
        print(f'\n{name.upper()} — no null values')

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 2: Orders Analysis

**Key Question:** What date range does the dataset cover? Any gaps or anomalies?

We examine order status distribution, monthly order volume, and the temporal span of the data.

In [ ]:
# Parse datetime columns
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Order status distribution
status_counts = orders['order_status'].value_counts()
print('Order status distribution:')
print(status_counts.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
status_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Order Status Distribution')
ax.set_xlabel('Status')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly order count over time
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
monthly_orders = orders.groupby('order_month').size()

print(f'Date range: {orders["order_purchase_timestamp"].min()} to {orders["order_purchase_timestamp"].max()}')
print(f'Total months: {len(monthly_orders)}')

fig, ax = plt.subplots(figsize=(12, 4))
monthly_orders.plot(kind='line', marker='o', linestyle='-', linewidth=1.5, ax=ax, color='coral')
ax.set_title('Monthly Order Volume')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
plt.tight_layout()
plt.show()

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 3: Customer Behavior

**Key Question:** Is the churn threshold of 180 days reasonable for this dataset?

We compute per-customer order frequency and total spend, then examine the distribution and repeat-purchase rate.

In [ ]:
# Merge orders with customers and items for per-customer metrics
cust_orders = orders.merge(customers, on='customer_id')
cust_spend = cust_orders.merge(items, on='order_id')

# Order frequency per customer
freq = cust_orders.groupby('customer_unique_id')['order_id'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of order frequency
freq.plot(kind='hist', bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Order Frequency per Customer')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Customer Count')

# Total spend per customer
spend = cust_spend.groupby('customer_unique_id')['price'].sum()
spend.plot(kind='hist', bins=50, ax=axes[1], color='steelblue', edgecolor='white', logy=True)
axes[1].set_title('Total Spend per Customer (log scale)')
axes[1].set_xlabel('Total Spend (BRL)')
axes[1].set_ylabel('Customer Count (log)')

plt.tight_layout()
plt.show()

In [ ]:
# % of customers with only 1 order vs 2+ orders
single_order = (freq == 1).sum()
repeat_orders = (freq >= 2).sum()
total_cust = len(freq)

print(f'Total unique customers: {total_cust}')
print(f'Single-order customers: {single_order} ({single_order/total_cust*100:.1f}%)')
print(f'Repeat customers (2+):  {repeat_orders} ({repeat_orders/total_cust*100:.1f}%)')

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 4: RFM Distributions

**Key Question:** Do the RFM distributions suggest natural customer segments? Any skew that needs transformation?

We compute Recency (days since last purchase), Frequency, and Monetary total per customer, then visualize distributions and correlations.

In [ ]:
# Build RFM table
snapshot_date = orders['order_purchase_timestamp'].max()
print(f'Snapshot date (max order timestamp): {snapshot_date}')

# Merge orders with items for monetary, with customers for unique_id
order_cust = orders.merge(customers, on='customer_id')
order_full = order_cust.merge(items[['order_id', 'price']], on='order_id')

rfm = order_full.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('price', 'sum')
).reset_index()

print(f'RFM table shape: {rfm.shape}')
rfm.head()

In [ ]:
# Plot distributions of RFM features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm['recency'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Recency (days since last order)')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Customer Count')

axes[1].hist(rfm['frequency'], bins=20, color='steelblue', edgecolor='white')
axes[1].set_title('Frequency (total orders)')
axes[1].set_xlabel('Orders')
axes[1].set_ylabel('Customer Count')

axes[2].hist(rfm['monetary'], bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('Monetary (total spend BRL)')
axes[2].set_xlabel('BRL')
axes[2].set_ylabel('Customer Count')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of RFM features
fig, ax = plt.subplots(figsize=(6, 5))
corr = rfm[['recency', 'frequency', 'monetary']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('RFM Feature Correlation')
plt.tight_layout()
plt.show()

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 5: Review Scores

**Key Question:** Is there any temporal drift in customer satisfaction?

We examine the distribution of review scores and their evolution over time.

In [ ]:
# Merge review scores with orders to get timestamps
rev_orders = reviews.merge(orders[['order_id', 'order_purchase_timestamp']], on='order_id')

# Distribution of review scores
score_counts = rev_orders['review_score'].value_counts().sort_index()
print('Review score distribution:')
print(score_counts.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
score_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Distribution of Review Scores')
ax.set_xlabel('Score')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Average review score over time (monthly)
rev_orders['order_month'] = rev_orders['order_purchase_timestamp'].dt.to_period('M')
monthly_score = rev_orders.groupby('order_month')['review_score'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
monthly_score.plot(kind='line', marker='o', linestyle='-', linewidth=1.5, ax=ax, color='coral')
ax.axhline(y=rev_orders['review_score'].mean(), color='gray', linestyle='--', alpha=0.7, label='Overall mean')
ax.set_title('Average Review Score by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Average Score')
ax.legend()
plt.tight_layout()
plt.show()

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 6: Churn Label Analysis

**Key Question:** Is the class imbalance severe enough to require resampling strategies?

We apply the 180-day churn threshold, examine class balance, and explore churn rate by geography.

In [ ]:
# Create churn labels using 180-day threshold
rfm['churn'] = (rfm['recency'] > 180).astype(int)

churn_rate = rfm['churn'].mean()
class_balance = rfm['churn'].value_counts(normalize=True)

print(f'Churn rate (>180 days without purchase): {churn_rate*100:.2f}%')
print(f'\nClass distribution:')
print(class_balance.to_string())
print(f'\nCounts:')
print(rfm['churn'].value_counts().to_string())

In [ ]:
# Churn rate by customer state
cust_state = customers[['customer_unique_id', 'customer_state']].drop_duplicates()
rfm_state = rfm.merge(cust_state, on='customer_unique_id')

state_churn = rfm_state.groupby('customer_state')['churn'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
state_churn.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Churn Rate by Customer State')
ax.set_xlabel('State')
ax.set_ylabel('Churn Rate')
ax.axhline(y=churn_rate, color='coral', linestyle='--', alpha=0.7, label=f'Overall: {churn_rate:.2%}')
ax.legend()
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Findings

*[Fill in after running the notebook with real data]*

---
## Section 7: Preliminary Anomaly Candidates

**Key Question:** Do the statistical outliers make business sense as anomalies, or are they just big customers?

We compute z-scores for monetary and frequency, flag extreme values, and inspect the top candidates.

In [ ]:
# Compute z-scores for monetary and frequency
rfm['monetary_z'] = (rfm['monetary'] - rfm['monetary'].mean()) / rfm['monetary'].std()
rfm['frequency_z'] = (rfm['frequency'] - rfm['frequency'].mean()) / rfm['frequency'].std()

# Flag customers with |z-score| > 3 as potential anomalies
rfm['anomaly_flag'] = (rfm['monetary_z'].abs() > 3) | (rfm['frequency_z'].abs() > 3)

n_anomalies = rfm['anomaly_flag'].sum()
print(f'Potential anomalies (|z| > 3): {n_anomalies} ({n_anomalies/len(rfm)*100:.2f}%)')

In [ ]:
# Show top 10 anomaly candidates
top_anomalies = rfm[rfm['anomaly_flag']].sort_values('monetary_z', ascending=False)

display_cols = ['customer_unique_id', 'recency', 'frequency', 'monetary',
                'monetary_z', 'frequency_z']
print('Top 10 anomaly candidates by monetary z-score:')
top_anomalies[display_cols].head(10)

In [ ]:
# Optional: merge with customer state to see geographic pattern of anomalies
anom_state = top_anomalies.merge(cust_state, on='customer_unique_id')
print('Anomaly counts by state:')
print(anom_state['customer_state'].value_counts().to_string())

## Findings

*[Fill in after running the notebook with real data]*

---
## Summary

*[After completing the analysis, summarize the key takeaways that will drive the downstream ML pipeline design: preprocessing needs, churn modeling approach, anomaly detection strategy, and segmentation potential.]*